# Laboratorio 05 — La columna que no existía

**Duración: 60 minutos.**

El laboratorio anterior cerró con una pregunta.

> El formato de los documentos tributarios **cambió**. Los que llegan desde 2026 traen
> dos campos que antes no existían: por qué canal se emitió el documento y en qué
> sucursal. Los de 2024 no los tienen y nunca los van a tener.
>
> Ya sabes que la tabla no se reescribe para cambiar **cómo** se guarda.
> **¿Pasará lo mismo con lo que se guarda?**

---

### Por qué esto le duele a la DGT

Los formularios cambian. Siempre. Se agrega un campo al documento tributario y hoy,
en su plataforma, eso significa reprocesar años de historia para que la tabla acepte
el formato nuevo: semanas de trabajo y el riesgo de romper lo que ya funcionaba.

Peor todavía: en Hive, agregar una columna a una tabla con historia era una de las
formas clásicas de **corromper los datos viejos en silencio**. La tabla leía por
posición, y a partir del cambio los valores aparecían corridos de columna.

### En la libreta

Desde marzo el almacenero también anota el teléfono del proveedor.

Las páginas viejas **no se rehacen**: quedan como estaban, con esa casilla en blanco.
Y él las sigue leyendo sin confundirse, porque sabe qué casilla es cuál — no cuenta
posiciones.

Y si un día decide que la casilla "teléfono" se llame mejor "contacto", cambia el
rótulo. Ninguna página se reescribe.

---

### Cómo se usa este notebook

Como siempre: `%%sql` en la primera línea, una sentencia por celda.

Hoy vas a ver **dos celdas que fallan a propósito**. No te asustes cuando pase: están
puestas ahí porque el error es el contenido. Cuando toque, se avisa.

**Recuerda el nombre a tres partes** para las vistas del sistema:
`mi_espacio.documentos_lab05.snapshots`.


## Paso 0 — Preparar la tabla

Primero, a tu espacio de trabajo.

In [ ]:
%%sql
-- Celda 0.1



Partimos de cero.

In [ ]:
%%sql
-- Celda 0.2



La tabla del laboratorio, con el **formato antiguo**: un año de documentos de
2024, tal como los recibía la DGT antes del cambio.

In [ ]:
%%sql
-- Celda 0.3



¿Cuántos quedaron?

In [ ]:
%%sql
-- Celda 0.4



Treinta mil. Ahora mira **qué forma tiene** la tabla: sus columnas y sus tipos.
Esto es lo que en Iceberg se llama el *esquema*.

In [ ]:
%%sql
-- Celda 0.5



## Paso 1 — Llega el formato nuevo

Diez columnas. Guárdalas en la cabeza, porque vamos a compararlas.

Ahora llega el lote de 2026, con el formato nuevo. Mírale la forma **antes** de
tocarlo.

In [ ]:
%%sql
-- Celda 1.1



**Doce columnas.** Las diez de siempre, y al final dos que tu tabla no tiene:
`canal_emision` y `codigo_sucursal`.

Vamos a intentar cargarlo tal cual, como haría cualquiera.

> **Esta celda va a fallar.** Está puesta a propósito: lo que hace el motor cuando le
> llega un lote con columnas de más es parte de lo que hay que saber.

In [ ]:
%%sql
-- Celda 1.2



## Paso 2 — Agregar las columnas

`too many data columns`: **demasiadas columnas en los datos**. El motor te dice
que la tabla tiene diez y le estás mandando doce, y se detiene.

Fíjate en lo que **no** hizo: no descartó columnas en silencio, no metió los valores
corridos, no cargó nada a medias. Falló, y falló entero. Para quien opera esto, un
error ruidoso es una buena noticia.

Así que hagamos lo que hay que hacer: agrandar la tabla. Las dos columnas de una sola
sentencia.

In [ ]:
%%sql
-- Celda 2.1



Listo, y fue instantáneo. **Ni un dato se movió**: lo único que cambió es la
descripción de la tabla.

Ahora la pregunta que importa. Los treinta mil documentos de 2024 se cargaron cuando
la tabla tenía diez columnas. ¿Qué pasa si los consulto ahora, que tiene doce?

In [ ]:
%%sql
-- Celda 2.2



**Ahí está la respuesta.** Los datos viejos están intactos —el RUT, el folio, el
monto, todo en su lugar— y las dos columnas nuevas vienen vacías, en `NULL`.

Vacías es la respuesta correcta: esos documentos de 2024 **nunca tuvieron** canal de
emisión. No es que se haya perdido; es que no existía.

Y esto es lo que en Hive se rompía. Ahí la tabla leía por posición, y agregar una
columna podía dejar los valores corridos: el monto apareciendo en la casilla del
estado. Iceberg identifica cada columna por un número interno propio, no por dónde
está ni por cómo se llama. Por eso no se confunde.

Confirmemos que no se perdió nada.

In [ ]:
%%sql
-- Celda 2.3



## Paso 3 — Cargar el lote nuevo

Treinta mil documentos, cero con canal. Todo en su lugar.

Ahora sí, el lote de 2026. **La misma sentencia que falló hace tres celdas**, sin
cambiarle una letra.

In [ ]:
%%sql
-- Celda 3.1



Entró.

Miremos las dos épocas juntas, en la misma tabla y en la misma consulta.

In [ ]:
%%sql
-- Celda 3.2



## Paso 4 — Los otros cambios

Dos formatos conviviendo sin ningún problema: 2024 sin canal, 2026 con canal, en
una sola tabla y respondiendo a la misma consulta.

Agregar columnas es lo más común, pero no es lo único. Veamos los otros cambios, uno
por celda, comprobando cada vez que los datos siguen bien.

**Renombrar.** Resulta que `codigo_sucursal` es un nombre incómodo y en la DGT todos
le dicen sucursal a secas.

In [ ]:
%%sql
-- Celda 4.1



Comprobemos que el dato sigue ahí, ahora bajo el nombre nuevo.

In [ ]:
%%sql
-- Celda 4.2



El mismo dato, otro rótulo. Ninguna página se reescribió.

**Ensanchar un tipo.** El código de tipo de documento se guardó como entero corto.
Supongamos que la DGT va a usar códigos más largos: se puede ampliar a entero
grande.

In [ ]:
%%sql
-- Celda 4.3



Y los valores que ya estaban, ¿siguen bien?

In [ ]:
%%sql
-- Celda 4.4



Todos en su lugar.

Ojo con la dirección: se puede **ensanchar** —de entero corto a largo, o un decimal
con más dígitos— porque todo lo que cabía antes cabe ahora. Al revés no: si pides
pasar de entero largo a corto, la respuesta es
`bigint cannot be cast to int`, y está bien que sea así.

**Eliminar.** Al final el código de sucursal no lo usaba nadie.

In [ ]:
%%sql
-- Celda 4.5



Y el resto de la tabla, intacto.

In [ ]:
%%sql
-- Celda 4.6



## Paso 5 — La historia del esquema

Cuatro cambios de forma a una tabla con treinta y dos mil documentos, y no se
reescribió un solo archivo.

La tabla, además, **deja registro**. Cada vez que cambia algo anota una versión nueva
de su descripción, y guarda la lista.

In [ ]:
%%sql
-- Celda 5.1



Cada fila es una versión de la descripción de la tabla, con la hora en que se
escribió. Y fíjate en la columna de la derecha: **pasa de 0 a 1**. El 0 es el formato
antiguo, de diez columnas; el 1 es la tabla ya con `canal_emision` y
`codigo_sucursal`.

> Un detalle de esta versión de Iceberg, para que no te desconcierte: **ese número no
> avanza con cada cambio**. Aquí hiciste cuatro y el registro solo muestra dos
> versiones. La lista completa de cambios no se lee de ahí; lo que sí se puede hacer
> —y es lo que vale— es **volver a ver la tabla como era**.

Y ahí conectamos con el laboratorio 01. Pidamos las páginas.

In [ ]:
%%sql
-- Celda 5.2



Dos páginas: la carga de 2024 y la de 2026.

**Copia el `snapshot_id` de la primera** —la de arriba, la de antes de todos los
cambios— y viajemos a ella.

In [ ]:
%%sql
-- Celda 5.3



Treinta mil: la tabla como estaba antes de que llegara el formato nuevo.

Y aquí viene lo bonito. Si en esa página la columna `canal_emision` **todavía no
existía**, ¿qué pasa si la pido?

> **Esta celda también va a fallar, a propósito.** Es la última.

In [ ]:
%%sql
-- Celda 5.4



---

## Listo

`Column 'canal_emision' does not exist`. **No existe**, porque en esa página de la
libreta esa casilla todavía no estaba.

La tabla no solo se acuerda de sus datos: se acuerda de la **forma** que tenía cuando
los guardó.

### Lo que te llevas

| El cambio | La sentencia | ¿Reescribe datos? |
|---|---|---|
| Agregar columnas | `ALTER TABLE … ADD COLUMNS (…)` | No |
| Renombrar una columna | `ALTER TABLE … RENAME COLUMN a TO b` | No |
| Ensanchar un tipo | `ALTER TABLE … ALTER COLUMN c TYPE BIGINT` | No |
| Eliminar una columna | `ALTER TABLE … DROP COLUMN c` | No |

Ninguno reescribe datos. Todos son instantáneos, en una tabla de treinta y dos mil
documentos y en una de doscientos millones.

**Y la certeza que hay que llevarse:** cambiar la forma de la tabla **no pone en
riesgo lo que ya está guardado**. Los documentos de 2024 sobrevivieron a cuatro
cambios de esquema, se siguen leyendo correctos, y las columnas que nunca tuvieron
aparecen vacías, que es exactamente lo que corresponde.

Para la DGT eso convierte un proyecto de semanas —reprocesar la historia cada vez que
cambia un formulario— en una sentencia.

### Dos bordes que conviene conocer

- **Los tipos se ensanchan, no se angostan.** De entero corto a largo sí; al revés
  no, y el motor lo dice: `bigint cannot be cast to int`.
- **Reordenar columnas no funciona en este ambiente.** `ALTER COLUMN … AFTER` y
  `… FIRST` fallan contra el catálogo, con un error que además avisa que no puede
  determinar si el cambio se aplicó o no. **No lo intentes en una tabla que te
  importe.** No es una limitación de Iceberg sino de cómo conversa con el catálogo de
  esta plataforma, y en la práctica se resuelve seleccionando las columnas en el orden
  que quieras al consultar.

### La pregunta del laboratorio siguiente

Ya sabes cargar sin duplicar, corregir, borrar, particionar y cambiar la forma de la
tabla.

Pero fíjate en algo: **todo eso entró directo a la tabla que consulta todo el mundo.**
Cuando cargaste el lote de 2026, Valentina podría haber estado consultando esa misma
tabla en ese preciso momento.

¿Y si el lote nuevo viene malo? ¿Si trae montos que no cuadran, o le falta la mitad de
los documentos? Ya se publicó.

**¿Cómo se revisa un lote antes de que lo vea nadie?** Eso es el **Laboratorio 06 —
Publicar solo si cuadra**.
